<a href="https://colab.research.google.com/github/yelinthant-kyoukasan/bed-sp-sem2/blob/main/automatic_model_training_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Quick-start:** If you just want to train a basic custom model for openWakeWord!

Follow the instructions for Step 1 below. Each time you change the wake word, click the play icon to the left of the title to generate a sample and make sure it sounds correct. The first time it takes ~30 seconds but subsequent runs will be quick.

Once you're satisfied with the pronounciation, go to the "Runtime" dropdown menu in the upper left of the page, and select "run all". Keep the tab open but feel free to do something else. After ~1 hour, your custom model will be ready and will automatically be downloaded to your computer!

If you are a Home Assistant user with the openWakeWord add-on, follow the instructions [here](https://github.com/home-assistant/addons/blob/master/openwakeword/DOCS.md#custom-wake-word-models) to install and enable your custom model.

---

If you are interested in learning more about the custom model training process (and increasing the accuracy of your custom models), read through each step in this notebook and try experimenting with different training parameters. If you have any questions or problems, feel free to start a discussion at the openWakeWord [repo](https://github.com/dscripka/openWakeWord/discussions).

## Training your own openWakeWord models


In [1]:
# @title  { display-mode: "form" }
# @markdown # 1. Test Example Training Clip Generation
# @markdown Since openWakeWord models are trained on synthetic examples of your
# @markdown target wake word, it's a good idea to make sure that the examples
# @markdown sound correct. Type in your target wake word below, and run the
# @markdown cell to listen to it.
# @markdown
# @markdown Here are some tips that can help get the wake word to sound right:

# @markdown - If your wake word isn't being pronounced in the way
# @markdown you want, try spelling out the sounds phonetically with underscores
# @markdown separating each part.
# @markdown For example: "hey siri" --> "hey_seer_e".

# @markdown - Spell out numbers ("2" --> "two")

# @markdown - Avoid all punctuation except for "?" and "!", and remove unicode characters

import os
import sys
import yaml
from IPython.display import Audio
if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

    # Install system dependencies
    !pip install piper-phonemize
    !pip install webrtcvad

    if "piper-sample-generator/" not in sys.path:
        sys.path.append("piper-sample-generator/")

    from generate_samples import generate_samples

target_word = "Com'pewta." # @param {type:"string"}

def text_to_speech(text):
    generate_samples(text = text,
                max_samples=1,
                length_scales=[1.1],
                noise_scales=[0.7], noise_scale_ws = [0.7],
                output_dir = './', batch_size=1, auto_reduce_batch_size=True,
                file_names=["test_generation.wav"]
                )

text_to_speech(target_word)
Audio("test_generation.wav", autoplay=True)


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 3.36 MiB/s, done.
Resolving deltas: 100% (93/93), done.
--2026-03-31 01:02:36--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-31T01%3A54%3A49Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-4

ModuleNotFoundError: No module named 'generate_samples'

In [2]:
# ============================================================================
# LKC MEDICINE — COMPLETE WAKE WORD TRAINING (All-in-One)
# Paste this entire cell into a fresh Google Colab notebook and run it.
# Everything is automated. Come back in ~1 hour and your model files
# will be downloaded automatically.
# ============================================================================

import os, sys
import numpy as np
import scipy.io.wavfile

# ===================== STEP 1: Install dependencies =====================
print("=" * 60)
print("STEP 1: Installing dependencies...")
print("=" * 60)

!pip install -q speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.1 \
  acoustics==0.2.6 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 \
  torchinfo==1.8.0 torchmetrics==1.2.0 mutagen==1.47.0 piper-tts onnxscript

# ===================== STEP 2: Patch torch_audiomentations =====================
print("\n" + "=" * 60)
print("STEP 2: Patching torch_audiomentations...")
print("=" * 60)

ta_io = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"
if os.path.exists(ta_io):
    with open(ta_io, 'r') as f:
        c = f.read()
    c = c.replace(
        'torchaudio.set_audio_backend("soundfile")',
        'try:\n    torchaudio.set_audio_backend("soundfile")\nexcept AttributeError:\n    pass'
    )
    with open(ta_io, 'w') as f:
        f.write(c)
    print("  Patched torchaudio.set_audio_backend!")
else:
    print("  File not found, will try after install")

# ===================== STEP 3: Clone repos =====================
print("\n" + "=" * 60)
print("STEP 3: Cloning repositories...")
print("=" * 60)

if not os.path.exists("./piper-sample-generator"):
    !git clone -q https://github.com/rhasspy/piper-sample-generator
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
      'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    print("  Piper sample generator cloned!")

if not os.path.exists("./openwakeword"):
    !git clone -q https://github.com/dscripka/openWakeWord.git openwakeword
    !pip install -q -e ./openwakeword
    print("  openWakeWord cloned and installed!")

# ===================== STEP 4: Download ONNX models =====================
print("\n" + "=" * 60)
print("STEP 4: Downloading ONNX models...")
print("=" * 60)

os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -nc -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx \
  -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -nc -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx \
  -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
print("  ONNX models downloaded!")

# ===================== STEP 5: Download training data =====================
print("\n" + "=" * 60)
print("STEP 5: Downloading training data (this takes ~15 min)...")
print("=" * 60)

import datasets

if not os.path.exists("./mit_rirs"):
    print("  Downloading room impulse responses...")
    os.makedirs("./mit_rirs", exist_ok=True)
    for row in datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True):
        name = row['audio']['path'].replace("/", "_")
        scipy.io.wavfile.write(os.path.join("./mit_rirs", name), 16000,
                               (row['audio']['array'] * 32767).astype(np.int16))
    print(f"  RIRs: {len(os.listdir('./mit_rirs'))} files")

if not os.path.exists("./fma"):
    print("  Downloading background music (~5 min)...")
    os.makedirs("./fma", exist_ok=True)
    fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
               .cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for cnt in range(1000):
        row = next(fma)
        scipy.io.wavfile.write(os.path.join("./fma", f"{cnt:06d}.wav"), 16000,
                               (row['audio']['array'] * 32767).astype(np.int16))
    print(f"  FMA: 1000 files")

from huggingface_hub import hf_hub_download

if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    print("  Downloading ACAV features (~17 GB, may take 5-10 min)...")
    hf_hub_download(repo_id="davidscripka/openwakeword_features",
                    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
                    repo_type="dataset", local_dir=".")
    print("  ACAV features downloaded!")

if not os.path.exists("./validation_set_features.npy"):
    print("  Downloading validation features...")
    hf_hub_download(repo_id="davidscripka/openwakeword_features",
                    filename="validation_set_features.npy",
                    repo_type="dataset", local_dir=".")
    print("  Validation features downloaded!")

# ===================== STEP 6: Patch train.py =====================
print("\n" + "=" * 60)
print("STEP 6: Applying all patches to train.py...")
print("=" * 60)

tp = "openwakeword/openwakeword/train.py"
with open(tp, 'r') as f:
    content = f.read()

# Patch 1: Fix generate_samples import (new piper repo structure)
content = content.replace(
    'from generate_samples import generate_samples',
    'from piper_sample_generator.__main__ import generate_samples'
)
print("  Patch 1: generate_samples import — done")

# Patch 2: Fix torch.load weights_only issue (PyTorch 2.6+)
content = content.replace(
    'import torch\n',
    'import torch\n_original_torch_load = torch.load\ntorch.load = lambda *args, **kwargs: _original_torch_load(*args, **{**kwargs, "weights_only": False})\n'
)
print("  Patch 2: torch.load weights_only — done")

# Patch 3: Fix torchaudio.info removed in newer versions
torchaudio_patch = """
import torchaudio as _ta
if not hasattr(_ta, 'info'):
    import soundfile as _sf
    class _AudioInfo:
        def __init__(self, sample_rate, num_frames):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
    def _torchaudio_info(filepath):
        info = _sf.info(str(filepath))
        return _AudioInfo(sample_rate=info.samplerate, num_frames=info.frames)
    _ta.info = _torchaudio_info
"""
content = content.replace('import torch\n', 'import torch\n' + torchaudio_patch + '\n', 1)
print("  Patch 3: torchaudio.info — done")

# Patch 4: Add model argument to all generate_samples calls
lines = content.split('\n')
patched_count = 0
for i, line in enumerate(lines):
    if 'generate_samples(' in line and 'model=' not in line:
        lines[i] = line.replace(
            'generate_samples(',
            'generate_samples(model="piper-sample-generator/models/en_US-libritts_r-medium.pt",'
        )
        patched_count += 1
content = '\n'.join(lines)
print(f"  Patch 4: generate_samples model arg — {patched_count} calls patched")

with open(tp, 'w') as f:
    f.write(content)
print("  All patches saved!")

# ===================== STEP 7: Create training config =====================
print("\n" + "=" * 60)
print("STEP 7: Creating training config...")
print("=" * 60)

import yaml

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config["target_phrase"] = ["ell_kay_see_medicine"]
config["model_name"] = "lkc_medicine"
config["n_samples"] = 3000
config["n_samples_val"] = 500
config["steps"] = 20000
config["output_dir"] = "./my_custom_model"
config["piper_sample_generator_path"] = "./piper-sample-generator"
config["background_paths"] = ["./fma"]
config["rir_paths"] = ["./mit_rirs"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["target_accuracy"] = 0.7
config["target_recall"] = 0.5
config["target_false_positives_per_hour"] = 0.2

with open("my_model.yaml", "w") as f:
    yaml.dump(config, f)
print(f"  Wake word: {config['target_phrase']}")
print(f"  Model name: {config['model_name']}")
print(f"  Training samples: {config['n_samples']}")
print(f"  Training steps: {config['steps']}")

# ===================== STEP 8: Generate clips =====================
print("\n" + "=" * 60)
print("STEP 8: Generating synthetic speech clips (~15 min)...")
print("=" * 60)

!python openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# ===================== STEP 8.5: Resample clips to 16kHz =====================
print("\n" + "=" * 60)
print("STEP 8.5: Resampling clips to 16kHz...")
print("=" * 60)

import glob, librosa

def resample_folder(folder_path):
    clips = glob.glob(f"{folder_path}/*.wav")
    count = 0
    for c in clips:
        sr, data = scipy.io.wavfile.read(c)
        if sr != 16000:
            audio = data.astype(np.float32) / 32767.0
            resampled = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            scipy.io.wavfile.write(c, 16000, (resampled * 32767).astype(np.int16))
            count += 1
    print(f"  Resampled {count}/{len(clips)} files in {os.path.basename(folder_path)}")

base = "my_custom_model/lkc_medicine"
for folder in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    path = os.path.join(base, folder)
    if os.path.exists(path):
        resample_folder(path)

print("  All clips resampled to 16kHz!")

# ===================== STEP 9: Augment clips =====================
print("\n" + "=" * 60)
print("STEP 9: Augmenting clips (~10 min)...")
print("=" * 60)

!python openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# ===================== STEP 10: Train model =====================
print("\n" + "=" * 60)
print("STEP 10: Training model (~20-30 min)...")
print("=" * 60)

!python openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# ===================== STEP 11: Download model files =====================
print("\n" + "=" * 60)
print("STEP 11: Downloading model files...")
print("=" * 60)

from google.colab import files

model_dir = "my_custom_model"
downloaded = []
for f in os.listdir(model_dir):
    filepath = os.path.join(model_dir, f)
    if os.path.isfile(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  Downloading: {f} ({size_kb:.1f} KB)")
        files.download(filepath)
        downloaded.append(f)

print("\n" + "=" * 60)
print("DONE! Downloaded files:")
for f in downloaded:
    print(f"  - {f}")
print("=" * 60)
print("Put lkc_medicine.onnx AND lkc_medicine.onnx.data into public/models/")

STEP 1: Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 37.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.2/805.2 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.0 MB/s e

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

  RIRs: 270 files


  FMA: 1000 files


...

<IPython.core.display.Javascript object>

  Downloading: lkc_medicine.onnx (13.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE! Downloaded files:
  - lkc_medicine.onnx.data
  - lkc_medicine.onnx
Put lkc_medicine.onnx AND lkc_medicine.onnx.data into public/models/


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 2. Download Data
# @markdown Training custom models requires downloading a wide variety of data
# @markdown that will help make the model perform well in real-world scenarios.
# @markdown This example notebook will download small samples of background noise,
# @markdown music, and Room Impulse Responses (to add echo). This will still produce
# @markdown a custom model that performs well, but if you are interested in adding even more,
# @markdown feel free to extend this notebook to download the full datasets and even add
# @markdown your own!
# @markdown
# @markdown Downloading this example data will usually take about 15 minutes.

# @markdown **Important note!** The data downloaded here has a mixture of difference
# @markdown licenses and usage restrictions. As such, any custom models trained with this
# @markdown data should be considered as appropriate for **non-commercial** personal use only.

# ## Install all dependencies
# !pip install datasets
# !pip install scipy
# !pip install tqdm

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip uninstall tensorflow -y
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models")
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# Imports
import sys

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

## Download all data

## Download MIR RIR data (takes about ~2 minutes)
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio (takes about ~3 minutes)

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xvf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

    # Save clips to 16-bit PCM wav files
    n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
    for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
        i += 1
        if i == n_hours*3600//30:
            break

# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    !wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
if not os.path.exists("validation_set_features.npy"):
    !wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy


Cloning into 'openwakeword'...
remote: Enumerating objects: 1181, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 1181 (delta 118), reused 151 (delta 99), pack-reused 935
Receiving objects: 100% (1181/1181), 3.29 MiB | 21.04 MiB/s, done.
Resolving deltas: 100% (707/707), done.
Obtaining file:///content/openwakeword
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.8 MB/s eta 0:00:00
  Building editable for openwakeword (pypr

100%|██████████| 270/270 [00:10<00:00, 24.55it/s]


--2024-04-08 01:49:11--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 65.8.178.118, 65.8.178.27, 65.8.178.93, ...
Connecting to huggingface.co (huggingface.co)|65.8.178.118|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2024-04-08 01:49:11 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 99%|█████████▉| 119/120 [00:39<00:00,  3.00it/s]


--2024-04-08 01:50:23--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 65.8.178.118, 65.8.178.93, 65.8.178.27, ...
Connecting to huggingface.co (huggingface.co)|65.8.178.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.huggingface.co/repos/a4/6f/a46f490589856ef0544c988c81f74c322707464d95ce7128c9df5f54295be163/721a66d0682c65a1b5c1da0aa109409cede1d20e28b15235c344b000cbb7654f?response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&Expires=1712800223&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcxMjgwMDIyM319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5odWdnaW5nZmFjZS5jby9yZXBvcy9hNC82Zi9hNDZmNDkwNTg5ODU2ZWYwNTQ0Yzk4OGM4MWY3NGMzMjI3MDc0NjRkOTVjZT

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 3. Train the Model
# @markdown Now that you have verified your target wake word and downloaded the data,
# @markdown the last step is to adjust the training paramaters (or keep
# @markdown the defaults below) and start the training!

# @markdown Each paramater controls a different aspect of training:
# @markdown - `number_of_examples` controls how many examples of your wakeword
# @markdown are generated. The default (1,000) usually produces a good model,
# @markdown but between 30,000 and 50,000 is often the best.

# @markdown - `number_of_training_steps` controls how long to train the model.
# @markdown Similar to the number of examples, the default (10,000) usually works well
# @markdown but training longer usually helps.

# @markdown - `false_activation_penalty` controls how strongly false activations
# @markdown are penalized during the training process. Higher values can make the model
# @markdown much less likely to activate when it shouldn't, but may also cause it
# @markdown to not activate when the wake word isn't spoken clearly and there is
# @markdown background noise.

# @markdown With the default values shown below,
# @markdown this takes about 30 - 60 minutes total on the normal CPU Colab runtime.
# @markdown If you want to train on more examples or train for longer,
# @markdown try changing the runtime type to a GPU to significantly speedup
# @markdown the example generating and model training.

# @markdown When the model finishes training, you can navigate to the `my_custom_model` folder
# @markdown in the file browser on the left (click on the folder icon), and download
# @markdown the [your target wake word].onnx or  <your target wake word>.tflite files.
# @markdown You can then use these as you would any other openWakeWord model!

# Load default YAML config file for training
import yaml
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Modify values in the config and save a new version
number_of_examples = 1000 # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 10000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500  # @param {type:"slider", min:100, max:5000, step:50}
config["target_phrase"] = [target_word]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = number_of_examples
config["n_samples_val"] = max(500, number_of_examples//10)
config["steps"] = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = false_activation_penalty

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Generate clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")

# Automatically download the trained model files
from google.colab import files

files.download(f"my_custom_model/{config['model_name']}.onnx")
files.download(f"my_custom_model/{config['model_name']}.tflite")


/usr/local/lib/python3.10/dist-packages/torch_audiomentations/utils/io.py:27: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:Batch 1/20 complete
DEBUG:generate_samples:Batch 2/20 complete
DEBUG:generate_samples:Batch 3/20 complete
DEBUG:generate_samples:Batch 4/20 complete
DEBUG:generate_samples:Batch 5/20 complete
DEBUG:generate_samples:Batch 6/20 complete
DEBUG:generate_samples:Batch 7/20 complete
DEBUG:generate_samples:Batch 8/20 complete
DEBUG:generate_samples:Batch 9/20 complete
DEBUG:generate_samples:Batch 10/20 c

/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're us

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>